In [2]:
import os
import zstandard  # pip install zstandard
from tqdm import tqdm
import random
import json
import langid
from typing import Generator, Optional, Set

files_processed_to_text = True




In [8]:
from datetime import datetime
def showTime():
    return str("["+datetime.now().strftime('%Y-%m-%d %H:%M:%S.%f')+" UTC]")

In [3]:
def zst_files_in_dir(directory):
    """List all .zst files in a directory."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".zst") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

In [4]:
def decompress_zst_to_text(
    input_file: str,
    vocab: Optional[Set[str]] = None,
    mode: str = "accuracy",
    ascii_threshold: float = 0.5
) -> Generator[str, None, None]:
    """
    Decompresses a .zst file containing JSONL (JSON lines) format, 
    and yields English texts filtered via language detection or ASCII checks.
    
    ### Parameters
    input_file (str):
        Path to the .zst file containing JSONL-formatted lines.
        
    vocab (Optional[Set[str]]):
        A vocabulary set to collect unique characters from valid English texts. Defaults to None.
        
    mode (str, default='accuracy'):
        - 'accuracy': Uses the `langid` library for precise English language detection.
        - 'speed': Uses an ASCII ratio check for faster filtering.
        
    ascii_threshold (float, default=0.5):
        Minimum ASCII character ratio (0.0-1.0) for mode='speed' to consider text as valid.
    
    ### Yields
    str:
        Filtered English text entries from the compressed file.
    """
    with open(input_file, "rb") as infile:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(infile) as reader:
            current_line = ""
            while True:
                chunk = reader.read(16384).decode("utf-8", errors="replace")  # Read in 16KB chunks
                if not chunk:
                    break
                current_line += chunk
                # Split into lines (handles partial lines)
                lines = current_line.split("\n")
                current_line = lines.pop() if lines else ""  # Save partial line for next iteration

                # Process each line
                for line in lines:
                    # Skip empty lines after stripping
                    stripped_line = line.strip()
                    if not stripped_line:
                        continue
                    
                    try:
                        data = json.loads(stripped_line)
                        text = data.get("text", "").strip()
                    except (json.JSONDecodeError, KeyError):
                        continue  # Skip invalid JSON
                        
                    except Exception as e:
                        print(f"JSON Error: {e} on line: {line[:50]}...")
                        continue

                    if mode == "accuracy":      
                        # Check language (English)
                        try:
                            detected_lang, _ = langid.classify(text)
                        except langid.langid.LanguageIdentificationError:
                            # Skip texts too short to identify
                            continue

                        if detected_lang != "en":
                            continue  # Non-English, skip

                    elif mode == "speed":
                        # ---- START FILTERING LOGIC ----
                        ascii_count = 0
                        total_chars = 0
                        
                        # Iterate through each character in text
                        for c in text:
                            code = ord(c)
                            if code <= 127:
                                ascii_count += 1
                            total_chars += 1

                        # Check filtering conditions
                        if total_chars == 0:
                            continue
                        if (ascii_count / total_chars) < ascii_threshold:
                            continue
                        # ---- END FILTERING LOGIC ----

                    # Update the vocabulary (only for English texts)
                    if vocab is not None:
                        vocab.update(set(text))

                    yield text.strip()


In [6]:
folder_path = "openwebtext2"
output_file = "output_v7_accuracy.txt"
vocab_file = "vocab_v7_accuracy.txt"


In [ ]:
# Gather files
files = zst_files_in_dir(folder_path)
total_files = len(files)
print(f"Total files: {total_files}")
print(files)
vocab = set()

In [ ]:
# Shuffle files randomly 
random.seed(42)  # Optional: Set seed for reproducibility
random.shuffle(files)  # Shuffle in-place
print(files)

In [8]:
# Process all files
if files_processed_to_text == False:
    with open(output_file, "w", encoding="utf-8") as outf:
        for filename in tqdm(files, total=len(files), desc="Processing Files"):
            print(f"{showTime()} Processing: {filename}")
            file_path = os.path.join(folder_path, filename)
            try:
                for text_line in decompress_zst_to_text(file_path, vocab, mode="accuracy"):
                    outf.write(text_line.strip())  # Write only the text line
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

In [9]:
# Write vocabulary
if files_processed_to_text == False:
    with open(vocab_file, "w", encoding="utf-8") as vfile:
        for char in sorted(vocab):
            vfile.write(char + "\n")

In [ ]:
#load sequence
with open(output_file, "r", encoding="utf-8") as f:
    number_of_characters_to_read = 10_000_000
    text_sequence = f.read(number_of_characters_to_read)

len(text_sequence)

In [23]:
# Karpathy minBPE repository
from minbpe import RegexTokenizer

tokenizer = RegexTokenizer()
tokenizer.train(text_sequence, vocab_size=16_384)

In [ ]:

tokenizer = RegexTokenizer()
tokenizer.train(text_sequence, vocab_size=16_384)

In [ ]:
vocab = tokenizer.vocab
vocab

In [ ]:
encoded_text = tokenizer.encode("Hello, world! I like apple juice - I drink it every day. Isn't that too much?")
print(encoded_text)

In [ ]:
decoded_text = tokenizer.decode(encoded_text)
print(decoded_text)

In [27]:
max_vocab_id = list(tokenizer.vocab.keys())[-1]
tokenizer.special_tokens = {
    "<|startoftext|>": max_vocab_id + 1,
    "<|separator|>": max_vocab_id + 2,
    "<|endoftext|>": max_vocab_id + 3,
    "<|unk|>": max_vocab_id + 4,
    "<|padding|>": max_vocab_id + 5
}

In [4]:
tokenizer_output_dir = "output_v7/tokenizer"
tokenizer_path = os.path.join(tokenizer_output_dir, "en_tokenizer")


In [ ]:
import os
if not os.path.exists(tokenizer_output_dir):
    os.makedirs(tokenizer_output_dir)


tokenizer.save(file_prefix=tokenizer_path)

Encoding the sequence of text

In [ ]:
# # Encoding the sequence of text

# from minbpe import RegexTokenizer

# tokenizer_output_dir = "output_v7/tokenizer"
# tokenizer_path = os.path.join(tokenizer_output_dir, "en_tokenizer")

# tokenizer = RegexTokenizer()
# tokenizer.load(model_file=tokenizer_path+".model")

In [ ]:
# # Encode the data in batches

# encoded_text_sequence = []
# batch_size = 100_000_000


# with open(output_file, "r", encoding="utf-8") as f:
#     while True:
#         chunk = f.read(batch_size)
#         if not chunk:
#             break
#         batch_tokens = tokenizer.encode(chunk)
#         encoded_text_sequence.extend(batch_tokens)
#         print(f"{showTime()} Processed {len(encoded_text_sequence)} tokens so far.")

# print(f"Total tokens: {len(encoded_text_sequence)}")

In [11]:
# import numpy as np


# encoder_output_dir = "output_v7/encoded_data"
# import os
# if not os.path.exists(encoder_output_dir):
#     os.makedirs(encoder_output_dir)

# output_path = os.path.join(encoder_output_dir, "encoded_output_v7_accuracy.npy")
# np.save(output_path, np.array(encoded_text_sequence, dtype=np.int64))

# # Free up memory
# del encoded_text_sequence

//////////////////////////////////////

In [ ]:
# import os
# import logging
# from time import time
# import math
# import numpy as np
# import gc
# from joblib import Parallel, delayed


# # ── Logging Configuration ──────────────────────────────────────────────
# logging.basicConfig(
#     level=logging.DEBUG,                           # change to DEBUG for more detail
#     format="%(asctime)s [%(levelname)s] %(message)s",
#     datefmt="%H:%M:%S"
# )
# logger = logging.getLogger(__name__)

# def showTime():
#     return f"[{time():.2f}]"

# # ── Configuration ─────────────────────────────────────────────────────
# FILE_PATH       = "output_val_v3.txt"
# TOKENIZER_DIR   = os.path.join("output_v7", "tokenizer")
# TOKENIZER_MODEL = os.path.join(TOKENIZER_DIR, "en_tokenizer.model")
# BATCH_SIZE      = 10_000_000   # bytes per chunk
# ENC_DIR         = os.path.join("output_v7", "encoded_data")
# OUTPUT_FILENAME = "encoded_output_val_v3.npy"

# os.makedirs(ENC_DIR, exist_ok=True)
# logger.info(f"Config: FILE_PATH={FILE_PATH}, BATCH_SIZE={BATCH_SIZE}, "
#             f"TOKENIZER_MODEL={TOKENIZER_MODEL}, ENC_DIR={ENC_DIR}")

# # ── Chunk Reader ──────────────────────────────────────────────────────
# def chunk_reader(fp, size):
#     """Yield fixed-size byte chunks from an open file."""
#     while True:
#         data = fp.read(size)
#         if not data:
#             logger.info("chunk_reader: reached EOF.")
#             break
#         logger.debug(f"chunk_reader: yielding chunk of {len(data)} bytes.")
#         yield data

# # ── Tokenizer Lazy-Loader ─────────────────────────────────────────────
# tokenizer = None
# def tokenize_chunk(chunk_bytes):
#     """
#     Decode bytes -> text -> tokens. Lazy-loads the tokenizer on first call
#     in each worker process.
#     """
#     global tokenizer
#     if tokenizer is None:
#         from minbpe import RegexTokenizer
#         tokenizer = RegexTokenizer()
#         tokenizer.load(model_file=TOKENIZER_MODEL)
#         logger.info(f"Worker {os.getpid()}: tokenizer initialized.")

#     logger.debug(f"Worker {os.getpid()}: received {len(chunk_bytes)} bytes.")
#     text = chunk_bytes.decode("utf-8", errors="ignore")
#     del chunk_bytes  # Free memory
#     tokens = tokenizer.encode(text)
#     logger.debug(f"Worker {os.getpid()}: produced {len(tokens)} tokens.")
#     del text  # Free memory
#     gc.collect() # Free memory
    
#     return tokens

# # ── Main Pipeline ──────────────────────────────────────────────────────
# def run_parallel_encoding():
#     logger.info("Starting parallel encoding in-notebook.")

#     # 1) Estimate number of tasks
#     file_size = os.path.getsize(FILE_PATH)
#     num_tasks = math.ceil(file_size / BATCH_SIZE)
#     logger.info(f"File size: {file_size} bytes, estimated {num_tasks} chunks of ~{BATCH_SIZE} bytes each.")

#     # 2) Estimate token capacity (heuristic: 1 byte ≈ 1 token)
#     est_tokens = file_size
#     logger.info(f"Estimated token capacity: {est_tokens}")

#     # 3) Pre-allocate memmap
#     memmap_path = os.path.join(ENC_DIR, OUTPUT_FILENAME)
#     mem = np.lib.format.open_memmap(
#         memmap_path, mode="w+", dtype=np.int64, shape=(est_tokens,)
#     )
#     logger.info(f"Created memmap at {memmap_path} with shape {mem.shape}")

#     # 4) Parallel tokenize & write directly to memmap, with periodic flushes
#     ptr = 0
#     chunk_i = 0
#     FLUSH_INTERVAL = 10  # flush every 10 chunks

#     n_jobs = min(max(os.cpu_count() // 2, 1), 8)
#     logger.info(f"Launching Joblib Parallel with n_jobs={n_jobs}")
#     with open(FILE_PATH, "rb") as f:
#         for tokens in Parallel(n_jobs=n_jobs, backend="loky", verbose=5)(
#             delayed(tokenize_chunk)(chunk)
#             for chunk in chunk_reader(f, BATCH_SIZE)
#         ):
#             L = len(tokens)
#             mem[ptr:ptr+L] = tokens
#             ptr += L
#             chunk_i += 1
#             logger.info(f"{showTime()} Written {ptr} tokens so far (chunk {chunk_i}/{num_tasks}).")

#             # Free memory
#             del tokens # Free memory
#             gc.collect() # Free memory

#             # periodic flush to limit OS page cache usage
#             if chunk_i % FLUSH_INTERVAL == 0:
#                 mem.flush()
#                 logger.info(f"{showTime()} Flushed memmap after {chunk_i} chunks.")

#     # 5) Final flush (no in-RAM copy)
#     mem.flush()
#     logger.info(f"{showTime()} Done encoding. Total tokens={ptr}. Data in {memmap_path}")

# # Run the pipeline
# run_parallel_encoding()

In [1]:
import os
import logging
from time import time
import math
import numpy as np
import gc
from joblib import Parallel, delayed

# ── Logging Configuration ──────────────────────────────────────────────
logging.basicConfig(
    level=logging.DEBUG,  # Change to DEBUG for more detail
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger(__name__)

def show_time():
    return f"[{time():.2f}]"

# ── Configuration ─────────────────────────────────────────────────────
FILE_PATH       = "output_val_v3.txt"
TOKENIZER_DIR   = os.path.join("output_v7", "tokenizer")
TOKENIZER_MODEL = os.path.join(TOKENIZER_DIR, "en_tokenizer.model")
BATCH_SIZE      = 10_000_000  # bytes per chunk
ENC_DIR         = os.path.join("output_v7", "encoded_data")
OUTPUT_FILENAME = "encoded_output_val_v3.npy"
n_jobs = min(max(os.cpu_count() // 2, 1), 8)
BATCH_GROUP_SIZE = n_jobs  # Number of chunks to process before writing to disk


os.makedirs(ENC_DIR, exist_ok=True)
logger.info(f"Config: FILE_PATH={FILE_PATH}, BATCH_SIZE={BATCH_SIZE}, "
            f"TOKENIZER_MODEL={TOKENIZER_MODEL}, ENC_DIR={ENC_DIR}")

# ── Chunk Reader ──────────────────────────────────────────────────────
def chunk_reader(fp, size):
    """Yield fixed-size byte chunks from an open file."""
    while True:
        data = fp.read(size)
        if not data:
            logger.info("chunk_reader: reached EOF.")
            break
        logger.debug(f"chunk_reader: yielding chunk of {len(data)} bytes.")
        yield data

# ── Tokenizer Lazy-Loader ─────────────────────────────────────────────
tokenizer = None
def tokenize_chunk(chunk_bytes):
    """
    Decode bytes -> text -> tokens. Lazy-loads the tokenizer on first call
    in each worker process.
    """
    global tokenizer
    if tokenizer is None:
        from minbpe import RegexTokenizer
        tokenizer = RegexTokenizer()
        tokenizer.load(model_file=TOKENIZER_MODEL)
        logger.info(f"Worker {os.getpid()}: tokenizer initialized.")

    logger.debug(f"Worker {os.getpid()}: received {len(chunk_bytes)} bytes.")
    text = chunk_bytes.decode("utf-8", errors="ignore")
    del chunk_bytes  # Free memory
    tokens = tokenizer.encode(text)
    logger.debug(f"Worker {os.getpid()}: produced {len(tokens)} tokens.")
    del text  # Free memory

    return tokens

# ── Main Pipeline ──────────────────────────────────────────────────────
def run_parallel_encoding():
    logger.info("Starting parallel encoding.")

    logger.info(f"Launching Joblib Parallel with n_jobs={n_jobs}")

    temp_files = []
    batch_tokens = []
    total_tokens = 0
    chunk_i = 0

    with open(FILE_PATH, "rb") as f:
        chunks = list(chunk_reader(f, BATCH_SIZE))
        total_chunks = len(chunks)
        logger.info(f"Total chunks to process: {total_chunks}")

        for i in range(0, total_chunks, BATCH_GROUP_SIZE):
            chunk_group = chunks[i:i + BATCH_GROUP_SIZE]
            logger.info(f"Processing chunk group {i // BATCH_GROUP_SIZE + 1}")

            tokens_list = Parallel(n_jobs=n_jobs, backend="loky", verbose=5)(
                delayed(tokenize_chunk)(chunk) for chunk in chunk_group
            )

            for tokens in tokens_list:
                batch_tokens.extend(tokens)
                total_tokens += len(tokens)
                chunk_i += 1
                logger.info(f"{show_time()} Processed chunk {chunk_i}/{total_chunks}. Total tokens so far: {total_tokens}")
                del tokens
                gc.collect()

            # Write batch to temporary file
            temp_file_path = os.path.join(ENC_DIR, f"temp_batch_{i // BATCH_GROUP_SIZE}.npy")
            np.save(temp_file_path, np.array(batch_tokens, dtype=np.int64))
            temp_files.append(temp_file_path)
            logger.info(f"{show_time()} Written batch {i // BATCH_GROUP_SIZE + 1} to {temp_file_path}")

            # Clear batch_tokens for next group
            batch_tokens.clear()
            gc.collect()

    # Concatenate all temporary files into the final output file
    output_path = os.path.join(ENC_DIR, OUTPUT_FILENAME)
    with open(output_path, 'wb') as outfile:
        for temp_file in temp_files:
            with open(temp_file, 'rb') as infile:
                outfile.write(infile.read())
            os.remove(temp_file)  # Remove temporary file after writing
            logger.info(f"{show_time()} Appended {temp_file} to {output_path} and removed temporary file.")

    logger.info(f"{show_time()} Done encoding. Total tokens: {total_tokens}. Data saved in {output_path}")

# Run the pipeline
if __name__ == "__main__":
    run_parallel_encoding()

11:03:58 [INFO] Config: FILE_PATH=output_val_v3.txt, BATCH_SIZE=10000000, TOKENIZER_MODEL=output_v7\tokenizer\en_tokenizer.model, ENC_DIR=output_v7\encoded_data
11:03:58 [INFO] Starting parallel encoding.
11:03:58 [INFO] Launching Joblib Parallel with n_jobs=8
11:03:58 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_reader: yielding chunk of 10000000 bytes.
11:03:59 [DEBUG] chunk_r